## Requirement 3: Primal-Dual Factorized Hedge in a Highly Non-Stationary Environment

The goal is to solve a repeated online bidding problem with multiple campaigns, campaign conflicts, and a total budget constraint.

At each round, the learner chooses a bid vector, observes the highest competing bid for every campaign, receives utility from the auctions it wins, and pays the corresponding first-price costs.

The algorithm has three main components:

1. **Primal learner:** a factorized Hedge learner with one weight for each campaign-bid pair. This avoids maintaining one weight for every full bid vector, whose number grows exponentially with the number of campaigns.

2. **Dual learner:** a projected gradient update for the budget multiplier $\lambda_t$, which penalizes expensive bids through a Lagrangian reward.

3. **Hard budget filter:** before sampling an action, the learner removes all bid vectors whose worst-case cost is larger than the remaining budget. This guarantees that the cumulative cost never exceeds the total budget.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

## 1. Highly Non-Stationary Environment

The environment produces a sequence of highest competing bids for all campaigns.

Every $K$ rounds, a new multivariate Gaussian distribution is sampled:

$$m_t \sim \mathcal{N}(\mu_k, \Sigma_k).$$

The sampled vector is clipped to the interval $[0, 1]^N$, so each component can be interpreted as the highest competing bid in one campaign.

Because the parameters $(\mu_k, \Sigma_k)$ change over time, the sequence is non-stationary. Smaller values of $K$ correspond to faster changes in the market.

In [ ]:
class RandomMultivariateGaussianEnvironment:
    def __init__(
        self,
        T,
        N,
        K,
        sigma_scale=0.025,
        seed=0,
        structure_seed=12345,
        base_means=None,
        mean_shift_scale=0.65,
        mean_jitter=0.012
    ):
        self.T = T
        self.N = N
        self.K = K
        self.sigma_scale = sigma_scale
        self.mean_shift_scale = mean_shift_scale
        self.mean_jitter = mean_jitter
        self.rng = np.random.default_rng(seed)
        self.structure_rng = np.random.default_rng(structure_seed)

        if base_means is None:
            self.base_means = np.array([0.08, 0.25, 0.45, 0.70, 0.92])[:N]
        else:
            self.base_means = np.array(base_means, dtype=float)

        self.m_sequence = np.zeros((T, N))
        self.regime_ids = np.zeros(T, dtype=int)
        self.regime_means = []
        self.regime_covariances = []

        self._generate_sequence()

    def _sample_gaussian_parameters(self, regime_idx):
        # Keep campaign means visibly separated. Low-shift profiles preserve the
        # ordering; medium/high profiles rotate the separated means so the best
        # campaigns can change after a regime boundary.
        rotated_means = np.roll(self.base_means, regime_idx % self.N)
        mu = (
            (1.0 - self.mean_shift_scale) * self.base_means
            + self.mean_shift_scale * rotated_means
        )
        mu += self.structure_rng.normal(0.0, self.mean_jitter, size=self.N)
        mu = np.clip(mu, 0.03, 0.97)

        # Low variance makes the regime means readable; uncertainty across trials
        # now comes from different realizations around the same fixed structure.
        A = self.structure_rng.normal(0.0, self.sigma_scale, size=(self.N, self.N))
        Sigma = A @ A.T + 1e-4 * np.eye(self.N)

        return mu, Sigma

    def _generate_sequence(self):
        regime = -1

        for t in range(self.T):
            if t % self.K == 0:
                regime += 1
                mu, Sigma = self._sample_gaussian_parameters(regime)
                self.regime_means.append(mu)
                self.regime_covariances.append(Sigma)

            m_t = self.rng.multivariate_normal(mu, Sigma)
            m_t = np.clip(m_t, 0.0, 1.0)

            self.m_sequence[t] = m_t
            self.regime_ids[t] = regime

        self.regime_means = np.array(self.regime_means)
        self.regime_covariances = np.array(self.regime_covariances)

    def get_competing_bids(self, t):
        return self.m_sequence[t]

## 2. Feasible Action Space

An action is a bid vector $b = (b_1, \ldots, b_N)$, where each bid $b_i$ is selected from the discrete bid set.

Without additional constraints, the number of possible bid vectors is $|B|^N$, which grows exponentially with the number of campaigns.

A campaign is considered active when $b_i > 0$.

The conflict graph imposes compatibility constraints: if two campaigns are connected by an edge, they cannot both be active in the same round. The function `generate_feasible_actions` enumerates only the bid vectors that satisfy this conflict constraint.


In [ ]:
def is_feasible_action(action, conflict_edges):
   
    active = action > 0

    for i, j in conflict_edges:
        if active[i] and active[j]:
            return False

    return True


def generate_feasible_actions(bid_set, N, conflict_edges):
    actions = []
    for action_tuple in product(bid_set, repeat=N):
        action = np.array(action_tuple, dtype=float)

        if is_feasible_action(action, conflict_edges):
            actions.append(action)

    return np.array(actions)

## 3. Utility and Cost

We use a first-price auction model.

For campaign $i$, the learner wins the auction if its bid is at least the highest competing bid:

$$b_i \ge m_i.$$

If the learner wins campaign $i$, it receives value $v_i$ and pays its own bid $b_i$.

Therefore, for a bid vector $b$ and market vector $m_t$, the utility is

$$u_t(b) = \sum_i (v_i - b_i) \mathbf{1}\{b_i \ge m_{t,i}\},$$

and the cost is

$$c_t(b) = \sum_i b_i \mathbf{1}\{b_i \ge m_{t,i}\}.$$

The worst-case cost of an action is $\sum_i b_i$, which happens if all selected bids win. This quantity is used later to enforce the hard budget constraint.


In [ ]:
def compute_utility_and_cost(action, m_t, values):
    wins = action >= m_t #create a Boolean 

    utility = np.sum((values - action) * wins)
    cost = np.sum(action * wins)

    return utility, cost

## 4. Primal-Dual Learner with Factorized Hedge

A standard Hedge algorithm over all feasible bid vectors would require one weight for each complete action. This is simple conceptually, but the number of actions can be exponential in the number of campaigns.

To avoid this, the implementation uses a factorized primal learner.

### 4.1 Primal Component: Factorized Hedge

Instead of storing a weight for every full bid vector, the learner stores one weight for each campaign-bid pair:

$$w_{i,b}.$$

Here, $i$ is the campaign and $b$ is one possible bid value from the discrete bid set.

A complete action is still a full bid vector $a = (b_1, \ldots, b_N)$, but its score is built from the campaign-level weights:

$$\text{score}(a) \propto \prod_i w_{i,b_i}.$$

Equivalently, the code computes this in log-space as

$$\log \text{score}(a) = \sum_i \log w_{i,b_i},$$

which is numerically more stable than directly multiplying many weights.

The learner then samples a feasible bid vector according to these factorized scores.

Because Requirement 3 allows full feedback, after observing the whole market vector $m_t$, the learner can evaluate the outcome of every campaign-bid arm. For each campaign $i$ and bid $b$, the arm receives the Lagrangian reward

$$(v_i - b)\mathbf{1}\{b \ge m_{t,i}\} - \lambda_t b\mathbf{1}\{b \ge m_{t,i}\}.$$

This reward is then used to update the corresponding Hedge weight.

### 4.2 Dual Component: Budget Multiplier

The dual variable $\lambda_t$ represents the pressure induced by the budget constraint. If the realized cost in a round is larger than the target average cost $\rho = B/T$, then $\lambda_t$ increases and future expensive bids are penalized more strongly.

The update is

$$\lambda_{t+1} = \Pi_{[0, 1/\rho]}\left(\lambda_t + \eta (c_t - \rho)\right).$$

The projection keeps the multiplier non-negative and prevents it from becoming excessively large.

### 4.3 Hard Budget Constraint

The dual update encourages the learner to reduce spending, but by itself it does not guarantee that the total budget is never exceeded.

To enforce the budget as a hard constraint, before sampling an action the learner filters the action set and only keeps bid vectors satisfying

$$\sum_i b_i \le \text{remaining budget}.$$

Since the realized cost is

$$\sum_i b_i \mathbf{1}\{b_i \ge m_{t,i}\},$$

it is always smaller than or equal to $\sum_i b_i$. Therefore, this conservative filter guarantees that the cumulative cost cannot exceed the total budget.

If no positive-bid action is affordable, the learner selects the zero-bid action, which has cost zero.


In [ ]:
class PrimalDualHedgeLearner:
     def __init__(
        self,
        actions,
        values,
        budget,
        T,
        hedge_eta=0.2,
        dual_eta=None,
        seed=0
    ):
        self.actions = actions
        self.values = np.array(values)

        self.budget = budget
        self.remaining_budget = budget
        self.T = T
        self.rho = budget / T

        self.hedge_eta = hedge_eta #learning rate of hedge weight
        self.dual_eta = dual_eta if dual_eta is not None else 1 / np.sqrt(T) # learning rate of dual variable

        self.rng = np.random.default_rng(seed)

        self.num_actions = actions.shape[0]

        # Individual arms are campaign/bid choices. A feasible action is a super arm.
        self.bid_set = np.unique(actions)
        self.num_campaigns = actions.shape[1]
        self.num_bids = self.bid_set.shape[0]
        self.bid_to_idx = {bid: idx for idx, bid in enumerate(self.bid_set)}
        self.action_bid_indices = np.array([
            [self.bid_to_idx[bid] for bid in action]
            for action in actions
        ])
        self.action_worst_case_costs = np.sum(actions, axis=1)
        zero_action_candidates = np.where(self.action_worst_case_costs == 0.0)[0]
        self.zero_action_idx = int(zero_action_candidates[0])
        self.arm_weights = np.ones((self.num_campaigns, self.num_bids))

        self.lambda_dual = 0.0
        self.lambda_max = 1.0 / self.rho

     def get_distribution(self, budget_feasible_only=True):
        if budget_feasible_only:
            feasible_mask = self.action_worst_case_costs <= self.remaining_budget + 1e-12
        else:
            feasible_mask = np.ones(self.num_actions, dtype=bool)

        if not np.any(feasible_mask):
            feasible_mask = np.zeros(self.num_actions, dtype=bool)
            feasible_mask[self.zero_action_idx] = True

        # Factorized Hedge: P(a) is proportional to product_i w_i(b_i).
        # We work in log-space to avoid numerical overflow.
        clipped_weights = np.maximum(self.arm_weights, 1e-300)
        log_weights = np.log(clipped_weights)
        action_log_scores = np.sum(
            log_weights[np.arange(self.num_campaigns), self.action_bid_indices],
            axis=1
        )

        masked_log_scores = np.full(self.num_actions, -np.inf)
        masked_log_scores[feasible_mask] = action_log_scores[feasible_mask]
        max_log_score = np.max(masked_log_scores[feasible_mask])
        action_scores = np.zeros(self.num_actions)
        action_scores[feasible_mask] = np.exp(
            masked_log_scores[feasible_mask] - max_log_score
        )

        total_score = np.sum(action_scores)
        if total_score == 0.0 or not np.isfinite(total_score):
            action_scores[feasible_mask] = 1.0
            total_score = np.sum(action_scores)

        return action_scores / total_score

     def choose_action(self):
        probs = self.get_distribution(budget_feasible_only=True)
        action_idx = self.rng.choice(self.num_actions, p=probs)
        action = self.actions[action_idx]

        return action_idx, action
     
     def update(self, chosen_idx, m_t):
        chosen_action = self.actions[chosen_idx]
        utility, cost = compute_utility_and_cost(
            chosen_action,
            m_t,
            self.values
        )

        self.remaining_budget = max(0.0, self.remaining_budget - cost)

        # Full feedback at the individual-arm level: after observing m_t, update each
        # campaign/bid arm, not the whole feasible action/super arm.
        arm_rewards = np.zeros((self.num_campaigns, self.num_bids))

        for campaign_idx in range(self.num_campaigns):
            for bid_idx, bid in enumerate(self.bid_set):
                win = bid >= m_t[campaign_idx]
                arm_utility = (self.values[campaign_idx] - bid) * win
                arm_cost = bid * win
                arm_rewards[campaign_idx, bid_idx] = (
                    arm_utility - self.lambda_dual * arm_cost
                )

        # Normalize per campaign so each campaign's bid arms compete with each other.
        for campaign_idx in range(self.num_campaigns):
            rewards = arm_rewards[campaign_idx]
            min_r = np.min(rewards)
            max_r = np.max(rewards)

            if max_r > min_r:
                normalized_rewards = (rewards - min_r) / (max_r - min_r)
            else:
                normalized_rewards = np.zeros_like(rewards)

            self.arm_weights[campaign_idx] *= np.exp(
                self.hedge_eta * normalized_rewards
            )
        
        self.lambda_dual += self.dual_eta * (cost - self.rho)
        self.lambda_dual = np.clip(
            self.lambda_dual,
            0.0,
            self.lambda_max
        )

        return utility, cost, self.lambda_dual
     


## 5. Experiment

The experiment runs the primal-dual factorized Hedge learner in the highly non-stationary environment.

The tracked quantities are:

- cumulative utility,
- cumulative cost,
- average utility and average cost,
- remaining budget,
- dual variable $\lambda_t$,
- regret against the best static budget-feasible action in hindsight.

A valid run should satisfy

$$\sum_{t=1}^T c_t \le B,$$

which is enforced by the hard budget filter in the action selection step.


In [ ]:
# UNCERTAINTY ESTIMATION: run many independent trials and plot a 95% CI
# Experiment parameters
T = 1000
N = 5
K = 100
K_values = [25, 50, 100, 200]
large_window_K_values = [200, 300, 400, 500]
n_trials = 50

bid_set = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
values = np.ones(N)
budget = 250.0
base_competing_bid_means = np.array([0.08, 0.25, 0.45, 0.70, 0.92])

environment_profiles = {
    "low": {
        "mean_shift_scale": 0.05,
        "mean_jitter": 0.008,
        "sigma_scale": 0.018,
    },
    "medium": {
        "mean_shift_scale": 0.65,
        "mean_jitter": 0.012,
        "sigma_scale": 0.022,
    },
    "high": {
        "mean_shift_scale": 1.00,
        "mean_jitter": 0.015,
        "sigma_scale": 0.026,
    },
}
# Same regime schedule/means/covariances across trials; each trial only changes
# the Gaussian realization and learner randomness.
base_structure_seed = 12345
selected_environment_profile = "medium"

# Campaign conflicts: campaigns connected by an edge cannot both be active.
# This simulated graph is denser than the first toy example, but it is not complete:
# some campaigns can still be active together in the same round.
conflict_edges = [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]

actions = generate_feasible_actions(
    bid_set=bid_set,
    N=N,
    conflict_edges=conflict_edges
)


def compute_action_outcomes(actions, env, values, T):
    market_sequence = np.array([env.get_competing_bids(t) for t in range(T)])
    utilities_per_round = np.zeros((actions.shape[0], T))
    costs_per_round = np.zeros((actions.shape[0], T))

    for t, m_t in enumerate(market_sequence):
        wins = actions >= m_t
        utilities_per_round[:, t] = np.sum((values - actions) * wins, axis=1)
        costs_per_round[:, t] = np.sum(actions * wins, axis=1)

    return utilities_per_round, costs_per_round


def compute_piecewise_static_oracle(actions, env, values, budget, T, K_value):
    # Piecewise-static benchmark: the oracle can choose a new fixed action
    # at each Gaussian regime change. The only budget constraint is global:
    # total oracle cost over the whole horizon must be <= budget.
    utilities_per_round, costs_per_round = compute_action_outcomes(
        actions,
        env,
        values,
        T
    )

    block_ranges = [
        (start_t, min(start_t + K_value, T))
        for start_t in range(0, T, K_value)
    ]
    num_blocks = len(block_ranges)
    num_actions = actions.shape[0]

    block_utilities = np.zeros((num_blocks, num_actions))
    block_costs = np.zeros((num_blocks, num_actions))

    for block_idx, (start_t, end_t) in enumerate(block_ranges):
        block_utilities[block_idx] = utilities_per_round[:, start_t:end_t].sum(axis=1)
        block_costs[block_idx] = costs_per_round[:, start_t:end_t].sum(axis=1)

    # Costs are multiples of the bid-grid step, so a small integer DP solves the
    # multiple-choice knapsack exactly: choose one action per block under total budget.
    positive_bids = np.unique(actions[actions > 0])
    if positive_bids.size > 1:
        cost_unit = np.min(np.diff(np.sort(positive_bids)))
    elif positive_bids.size == 1:
        cost_unit = positive_bids[0]
    else:
        cost_unit = 1.0

    budget_units = int(np.floor(budget / cost_unit + 1e-9))
    block_cost_units = np.rint(block_costs / cost_unit).astype(int)

    dp = np.full((num_blocks + 1, budget_units + 1), -np.inf)
    prev_budget = np.full((num_blocks + 1, budget_units + 1), -1, dtype=int)
    prev_action = np.full((num_blocks + 1, budget_units + 1), -1, dtype=int)
    dp[0, 0] = 0.0

    for block_idx in range(num_blocks):
        for action_idx in range(num_actions):
            action_cost = block_cost_units[block_idx, action_idx]
            action_utility = block_utilities[block_idx, action_idx]

            if action_cost > budget_units:
                continue

            candidate = dp[block_idx, :budget_units + 1 - action_cost] + action_utility
            target = dp[block_idx + 1, action_cost:]
            improved = candidate > target

            if np.any(improved):
                improved_budgets = np.where(improved)[0] + action_cost
                dp[block_idx + 1, improved_budgets] = candidate[improved]
                prev_budget[block_idx + 1, improved_budgets] = improved_budgets - action_cost
                prev_action[block_idx + 1, improved_budgets] = action_idx

    best_budget_units = int(np.nanargmax(dp[num_blocks]))
    best_total_utility = dp[num_blocks, best_budget_units]

    block_oracle_indices = [0] * num_blocks
    current_budget = best_budget_units
    for block_idx in range(num_blocks, 0, -1):
        action_idx = prev_action[block_idx, current_budget]
        if action_idx < 0:
            action_idx = 0
        block_oracle_indices[block_idx - 1] = action_idx
        current_budget = prev_budget[block_idx, current_budget]
        if current_budget < 0:
            current_budget = 0

    oracle_utilities_over_time = np.zeros(T)
    oracle_costs_over_time = np.zeros(T)
    block_oracle_actions = []

    for block_idx, (start_t, end_t) in enumerate(block_ranges):
        action_idx = block_oracle_indices[block_idx]
        block_oracle_actions.append(actions[action_idx])
        oracle_utilities_over_time[start_t:end_t] = utilities_per_round[
            action_idx,
            start_t:end_t
        ]
        oracle_costs_over_time[start_t:end_t] = costs_per_round[
            action_idx,
            start_t:end_t
        ]

    return {
        "block_oracle_actions": block_oracle_actions,
        "block_oracle_indices": block_oracle_indices,
        "best_oracle_action": block_oracle_actions[-1],
        "best_oracle_utility": oracle_utilities_over_time.sum(),
        "best_oracle_cost": oracle_costs_over_time.sum(),
        "oracle_utilities_over_time": oracle_utilities_over_time,
        "oracle_costs_over_time": oracle_costs_over_time,
        "cumulative_oracle_utility": np.cumsum(oracle_utilities_over_time),
        "cumulative_oracle_cost": np.cumsum(oracle_costs_over_time),
        "oracle_remaining_budget": budget - np.cumsum(oracle_costs_over_time),
    }


def run_primal_dual_trial(env_seed, learner_seed, K_value=K, environment_profile=selected_environment_profile, structure_seed=base_structure_seed):
    profile = environment_profiles[environment_profile]
    env = RandomMultivariateGaussianEnvironment(
        T=T,
        N=N,
        K=K_value,
        sigma_scale=profile["sigma_scale"],
        seed=env_seed,
        structure_seed=structure_seed,
        base_means=base_competing_bid_means,
        mean_shift_scale=profile["mean_shift_scale"],
        mean_jitter=profile["mean_jitter"]
    )

    learner = PrimalDualHedgeLearner(
        actions=actions,
        values=values,
        budget=budget,
        T=T,
        hedge_eta=0.2,
        seed=learner_seed
    )

    utilities = np.zeros(T)
    costs = np.zeros(T)
    lambdas = np.zeros(T)
    remaining_budgets = np.zeros(T)
    chosen_actions = np.zeros((T, N))

    for t in range(T):
        chosen_idx, action = learner.choose_action()
        m_t = env.get_competing_bids(t)
        utility, cost, lambda_dual = learner.update(chosen_idx, m_t)

        utilities[t] = utility
        costs[t] = cost
        lambdas[t] = lambda_dual
        remaining_budgets[t] = learner.remaining_budget
        chosen_actions[t] = action

    cumulative_utility = np.cumsum(utilities)
    cumulative_cost = np.cumsum(costs)
    average_utility = cumulative_utility / np.arange(1, T + 1)
    average_cost = cumulative_cost / np.arange(1, T + 1)

    oracle = compute_piecewise_static_oracle(actions, env, values, budget, T, K_value)
    regret = oracle["cumulative_oracle_utility"] - cumulative_utility

    return {
        "env": env,
        "learner": learner,
        "utilities": utilities,
        "costs": costs,
        "lambdas": lambdas,
        "remaining_budgets": remaining_budgets,
        "chosen_actions": chosen_actions,
        "cumulative_utility": cumulative_utility,
        "cumulative_cost": cumulative_cost,
        "average_utility": average_utility,
        "average_cost": average_cost,
        "regret": regret,
        **oracle,
    }


trial_results = [
    run_primal_dual_trial(env_seed=1000 + trial, learner_seed=2000 + trial, K_value=K, structure_seed=base_structure_seed)
    for trial in range(n_trials)
]

regret_per_trial = np.array([result["regret"] for result in trial_results])
final_regret_per_trial = regret_per_trial[:, -1]
average_regret = regret_per_trial.mean(axis=0)
regret_sd = regret_per_trial.std(axis=0, ddof=1)
regret_se = regret_sd / np.sqrt(n_trials)
regret_ci95 = 1.96 * regret_se
rounds = np.arange(1, T + 1)

normalized_regret_per_trial = regret_per_trial / rounds
average_normalized_regret = normalized_regret_per_trial.mean(axis=0)
normalized_regret_sd = normalized_regret_per_trial.std(axis=0, ddof=1)
normalized_regret_ci95 = 1.96 * normalized_regret_sd / np.sqrt(n_trials)

# Keep the last run available for the other diagnostic plots.
last_trial = trial_results[-1]
env = last_trial["env"]
learner = last_trial["learner"]
utilities = last_trial["utilities"]
costs = last_trial["costs"]
lambdas = last_trial["lambdas"]
remaining_budgets = last_trial["remaining_budgets"]
chosen_actions = last_trial["chosen_actions"]
cumulative_utility = last_trial["cumulative_utility"]
cumulative_cost = last_trial["cumulative_cost"]
average_utility = last_trial["average_utility"]
average_cost = last_trial["average_cost"]
regret = last_trial["regret"]
cumulative_oracle_utility = last_trial["cumulative_oracle_utility"]
cumulative_oracle_cost = last_trial["cumulative_oracle_cost"]
oracle_remaining_budget = last_trial["oracle_remaining_budget"]
best_oracle_action = last_trial["best_oracle_action"]
best_oracle_utility = last_trial["best_oracle_utility"]
best_oracle_cost = last_trial["best_oracle_cost"]

print(f"Number of feasible actions: {actions.shape[0]}")
print(f"Number of trials: {n_trials}")
print(f"Mean final regret vs piecewise-static oracle: {final_regret_per_trial.mean():.3f}")
print(f"Final regret standard deviation: {final_regret_per_trial.std(ddof=1):.3f}")
print(f"Final regret 95% CI for the mean: {final_regret_per_trial.mean():.3f} +/- {1.96 * final_regret_per_trial.std(ddof=1) / np.sqrt(n_trials):.3f}")
print(f"Last-trial utility: {cumulative_utility[-1]:.3f}")
print(f"Last-trial cost: {cumulative_cost[-1]:.3f}")
print(f"Budget: {budget:.3f}")
print(f"Last-trial remaining budget: {remaining_budgets[-1]:.3f}")
print(f"Last-trial final lambda: {lambdas[-1]:.3f}")
print(f"Last-trial piecewise oracle utility: {best_oracle_utility:.3f}")
print(f"Last-trial piecewise oracle cost: {best_oracle_cost:.3f}")
print(f"Last-trial final regret: {regret[-1]:.3f}")
print(f"Last-trial final-block oracle action: {best_oracle_action}")

# Sensitivity test: repeat the uncertainty estimate for different regime lengths K.
k_sensitivity_results = {}
for K_test in K_values:
    K_trial_results = [
        run_primal_dual_trial(
            env_seed=10_000 + 100 * K_test + trial,
            learner_seed=20_000 + 100 * K_test + trial,
            K_value=K_test,
            structure_seed=base_structure_seed + K_test
        )
        for trial in range(n_trials)
    ]

    K_regret_per_trial = np.array([result["regret"] for result in K_trial_results])
    K_average_regret = K_regret_per_trial.mean(axis=0)
    K_regret_sd = K_regret_per_trial.std(axis=0, ddof=1)
    K_regret_ci95 = 1.96 * K_regret_sd / np.sqrt(n_trials)
    K_final_regrets = K_regret_per_trial[:, -1]

    k_sensitivity_results[K_test] = {
        "trial_results": K_trial_results,
        "regret_per_trial": K_regret_per_trial,
        "average_regret": K_average_regret,
        "regret_ci95": K_regret_ci95,
        "final_regrets": K_final_regrets,
        "mean_final_regret": K_final_regrets.mean(),
        "final_regret_ci95": 1.96 * K_final_regrets.std(ddof=1) / np.sqrt(n_trials),
    }

print("\nK sensitivity summary:")
for K_test, result in k_sensitivity_results.items():
    print(
        f"K={K_test:>3}: final regret "
        f"{result['mean_final_regret']:.3f} +/- {result['final_regret_ci95']:.3f}"
    )

# Large-window diagnostic requested after checking the late regret spike.
# It tests whether the regret increase after round 800 aligns with budget exhaustion.
large_window_results = {}
for K_test in large_window_K_values:
    if K_test in k_sensitivity_results:
        K_trial_results = k_sensitivity_results[K_test]["trial_results"]
    else:
        K_trial_results = [
            run_primal_dual_trial(
                env_seed=30_000 + 100 * K_test + trial,
                learner_seed=40_000 + 100 * K_test + trial,
                K_value=K_test,
                structure_seed=base_structure_seed + K_test
            )
            for trial in range(n_trials)
        ]

    K_regret_per_trial = np.array([result["regret"] for result in K_trial_results])
    K_remaining_budget_per_trial = np.array([
        result["remaining_budgets"]
        for result in K_trial_results
    ])
    K_cumulative_cost_per_trial = np.array([
        result["cumulative_cost"]
        for result in K_trial_results
    ])
    K_oracle_cost_per_trial = np.array([
        result["cumulative_oracle_cost"]
        for result in K_trial_results
    ])
    K_oracle_remaining_budget_per_trial = np.array([
        result["oracle_remaining_budget"]
        for result in K_trial_results
    ])

    K_average_regret = K_regret_per_trial.mean(axis=0)
    K_regret_ci95 = 1.96 * K_regret_per_trial.std(axis=0, ddof=1) / np.sqrt(n_trials)
    K_average_remaining_budget = K_remaining_budget_per_trial.mean(axis=0)
    K_average_cumulative_cost = K_cumulative_cost_per_trial.mean(axis=0)
    K_average_oracle_cost = K_oracle_cost_per_trial.mean(axis=0)
    K_average_oracle_remaining_budget = K_oracle_remaining_budget_per_trial.mean(axis=0)

    budget_exhaustion_rounds = []
    for remaining_budget_path in K_remaining_budget_per_trial:
        exhausted_rounds = np.where(remaining_budget_path <= 1e-9)[0]
        if exhausted_rounds.size > 0:
            budget_exhaustion_rounds.append(exhausted_rounds[0] + 1)
        else:
            budget_exhaustion_rounds.append(T)
    budget_exhaustion_rounds = np.array(budget_exhaustion_rounds)

    spike_start = min(800, T - 1)
    regret_increase_after_800 = K_regret_per_trial[:, -1] - K_regret_per_trial[:, spike_start]

    large_window_results[K_test] = {
        "trial_results": K_trial_results,
        "regret_per_trial": K_regret_per_trial,
        "average_regret": K_average_regret,
        "regret_ci95": K_regret_ci95,
        "average_remaining_budget": K_average_remaining_budget,
        "average_cumulative_cost": K_average_cumulative_cost,
        "average_oracle_cost": K_average_oracle_cost,
        "average_oracle_remaining_budget": K_average_oracle_remaining_budget,
        "budget_exhaustion_rounds": budget_exhaustion_rounds,
        "mean_budget_exhaustion_round": budget_exhaustion_rounds.mean(),
        "remaining_budget_at_800": K_remaining_budget_per_trial[:, spike_start],
        "mean_remaining_budget_at_800": K_remaining_budget_per_trial[:, spike_start].mean(),
        "regret_increase_after_800": regret_increase_after_800,
        "mean_regret_increase_after_800": regret_increase_after_800.mean(),
        "final_regrets": K_regret_per_trial[:, -1],
    }

print("\nLarge-window budget-spike diagnostic:")
for K_test, result in large_window_results.items():
    print(
        f"K={K_test:>3}: final regret={result['final_regrets'].mean():.3f}, "
        f"regret increase after round 800={result['mean_regret_increase_after_800']:.3f}, "
        f"remaining budget at round 800={result['mean_remaining_budget_at_800']:.3f}, "
        f"mean budget exhaustion round={result['mean_budget_exhaustion_round']:.1f}"
    )

### Environment profiles

The competing-bid means are deliberately separated across campaigns: `[0.08, 0.25, 0.45, 0.70, 0.92]`. The regime boundaries are fixed, and the same regime means are reused across trials. Low shift keeps the campaign ordering almost unchanged, while medium and high shift rotate the separated means so that the best campaigns can change more visibly after a regime boundary.

In [ ]:
# Visualize low/medium/high environment profiles with the same fixed regime structure.
env_profile_examples = {
    name: RandomMultivariateGaussianEnvironment(
        T=T,
        N=N,
        K=K,
        seed=123,
        structure_seed=base_structure_seed,
        base_means=base_competing_bid_means,
        **profile
    )
    for name, profile in environment_profiles.items()
}

fig, axes = plt.subplots(1, len(env_profile_examples), figsize=(14, 4), sharex=True, sharey=True)
if len(env_profile_examples) == 1:
    axes = [axes]

campaign_colors = plt.cm.tab10(np.arange(N))
for ax, (profile_name, profile_env) in zip(axes, env_profile_examples.items()):
    for campaign_idx in range(N):
        ax.step(
            np.arange(T),
            profile_env.regime_means[profile_env.regime_ids, campaign_idx],
            where="post",
            linewidth=2,
            color=campaign_colors[campaign_idx],
            label=f"campaign {campaign_idx + 1}"
        )

    for boundary in range(K, T, K):
        ax.axvline(boundary, color="black", linestyle="--", linewidth=0.8, alpha=0.30)

    ax.set_title(f"{profile_name.capitalize()} shift")
    ax.set_xlabel("Round")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.20)

axes[0].set_ylabel("Regime mean of competing bid")
axes[-1].legend(ncol=1, fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.suptitle("Fixed environment structure: separated campaign means", y=1.03)
plt.tight_layout()
plt.show()

for profile_name, profile_env in env_profile_examples.items():
    first_three = np.round(profile_env.regime_means[:3], 3)
    print(f"{profile_name} first 3 regime means:\n{first_three}\n")

### Sensitivity to K

`K` controls how many rounds pass before the Gaussian market distribution changes. Smaller `K` means a faster-changing, more non-stationary environment. The plot below compares the average cumulative regret for several `K` values, with 95% confidence intervals over trials.


In [ ]:
# Compare uncertainty across different K values
rounds = np.arange(1, T + 1)

plt.figure(figsize=(9, 5))
for K_test, result in k_sensitivity_results.items():
    avg = result["average_regret"]
    ci95 = result["regret_ci95"]
    plt.plot(rounds, avg, label=f"K={K_test}")
    plt.fill_between(rounds, avg - ci95, avg + ci95, alpha=0.15)

plt.axhline(0.0, color="black", linewidth=1)
plt.title("Cumulative regret for different K values")
plt.xlabel("Round")
plt.ylabel("Regret")
plt.legend(title="Gaussian window")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
K_labels = list(k_sensitivity_results.keys())
mean_final_regrets = [
    k_sensitivity_results[K_test]["mean_final_regret"]
    for K_test in K_labels
]
final_regret_ci95 = [
    k_sensitivity_results[K_test]["final_regret_ci95"]
    for K_test in K_labels
]

plt.errorbar(
    K_labels,
    mean_final_regrets,
    yerr=final_regret_ci95,
    marker="o",
    capsize=4,
    linewidth=2
)
plt.title("Final regret sensitivity to K")
plt.xlabel("K: rounds before Gaussian changes")
plt.ylabel("Final regret")
plt.xticks(K_labels)
plt.tight_layout()
plt.show()


### Large-window and budget-spike diagnostic

This diagnostic answers whether the late regret spike is associated with the learner running out of budget. It repeats the experiment for larger Gaussian windows `K = 200, 300, 400, 500`, then compares cumulative regret, remaining budget, cumulative learner cost, and cumulative oracle cost.


In [ ]:
# Large-window diagnostic: regret pattern and budget exhaustion
rounds = np.arange(1, T + 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for K_test, result in large_window_results.items():
    avg_regret = result["average_regret"]
    ci95 = result["regret_ci95"]
    axes[0, 0].plot(rounds, avg_regret, label=f"K={K_test}")
    axes[0, 0].fill_between(rounds, avg_regret - ci95, avg_regret + ci95, alpha=0.10)

axes[0, 0].axhline(0.0, color="black", linewidth=1)
axes[0, 0].axvline(800, color="red", linestyle="--", linewidth=1, label="round 800")
axes[0, 0].set_title("Cumulative regret for larger K values")
axes[0, 0].set_xlabel("Round")
axes[0, 0].set_ylabel("Regret")
axes[0, 0].legend()

for K_test, result in large_window_results.items():
    axes[0, 1].plot(rounds, result["average_remaining_budget"], label=f"Learner K={K_test}")
    axes[0, 1].plot(rounds, result["average_oracle_remaining_budget"], linestyle="--", label=f"Oracle K={K_test}")

axes[0, 1].axhline(0.0, color="black", linewidth=1)
axes[0, 1].axvline(800, color="red", linestyle="--", linewidth=1)
axes[0, 1].set_title("Average remaining budget: learner vs oracle")
axes[0, 1].set_xlabel("Round")
axes[0, 1].set_ylabel("Remaining budget")
axes[0, 1].legend(fontsize=8, ncol=2)

for K_test, result in large_window_results.items():
    axes[1, 0].plot(rounds, result["average_cumulative_cost"], label=f"Learner K={K_test}")
    axes[1, 0].plot(rounds, result["average_oracle_cost"], linestyle="--", label=f"Oracle K={K_test}")

axes[1, 0].axhline(budget, color="black", linewidth=1, label="budget")
axes[1, 0].axvline(800, color="red", linestyle="--", linewidth=1)
axes[1, 0].set_title("Average cumulative cost: learner vs oracle")
axes[1, 0].set_xlabel("Round")
axes[1, 0].set_ylabel("Cumulative cost")
axes[1, 0].legend(fontsize=8, ncol=2)

K_labels = list(large_window_results.keys())
regret_spikes = [
    large_window_results[K_test]["mean_regret_increase_after_800"]
    for K_test in K_labels
]
remaining_at_800 = [
    large_window_results[K_test]["mean_remaining_budget_at_800"]
    for K_test in K_labels
]
exhaustion_rounds = [
    large_window_results[K_test]["mean_budget_exhaustion_round"]
    for K_test in K_labels
]

x = np.arange(len(K_labels))
width = 0.35
axes[1, 1].bar(x - width / 2, regret_spikes, width, label="Regret increase after 800")
axes[1, 1].bar(x + width / 2, remaining_at_800, width, label="Remaining budget at 800")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(K_labels)
axes[1, 1].set_title("Late spike vs remaining budget")
axes[1, 1].set_xlabel("K")
axes[1, 1].legend()

for idx, exhaustion_round in enumerate(exhaustion_rounds):
    axes[1, 1].text(
        idx,
        max(regret_spikes[idx], remaining_at_800[idx]) + 2,
        f"exh. {exhaustion_round:.0f}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

print("Large-window diagnostic interpretation:")
for K_test, result in large_window_results.items():
    print(
        f"K={K_test}: after round 800 regret increases by "
        f"{result['mean_regret_increase_after_800']:.2f} on average; "
        f"remaining budget at round 800 is "
        f"{result['mean_remaining_budget_at_800']:.2f}; "
        f"mean budget exhaustion round is "
        f"{result['mean_budget_exhaustion_round']:.1f}."
    )


### Uncertainty estimate

This plot uses `n_trials = 50` independent runs. The solid line is the mean cumulative regret against the piecewise-static oracle and the shaded area is the 95% confidence interval of the mean.


In [ ]:
# Standalone uncertainty plot
rounds = np.arange(1, T + 1)

plt.figure(figsize=(8, 5))
plt.plot(rounds, average_regret, label="Estimated regret vs piecewise oracle")
plt.fill_between(
    rounds,
    average_regret - regret_ci95,
    average_regret + regret_ci95,
    alpha=0.25,
    label="Uncertainty: 95% CI"
)
plt.plot(rounds, regret, alpha=0.7, label="Last trial regret")
plt.axhline(0.0, color="black", linewidth=1)
plt.title("Cumulative regret with uncertainty")
plt.xlabel("Round")
plt.ylabel("Regret")
plt.legend()
plt.tight_layout()
plt.show()


### Normalized regret

Cumulative regret can keep growing because it adds the utility gap over time. To check whether the average gap per round stabilizes, we also plot normalized regret, `R_t / t`, with a 95% confidence interval over trials.


In [ ]:
# Normalized regret plot: R_t / t
rounds = np.arange(1, T + 1)

plt.figure(figsize=(8, 5))
plt.plot(rounds, average_normalized_regret, label="Normalized regret vs piecewise oracle")
plt.fill_between(
    rounds,
    average_normalized_regret - normalized_regret_ci95,
    average_normalized_regret + normalized_regret_ci95,
    alpha=0.25,
    label="Uncertainty: 95% CI"
)
plt.axhline(0.0, color="black", linewidth=1)
plt.title("Normalized regret vs piecewise-static oracle, K=100")
plt.xlabel("Round")
plt.ylabel("Regret / round")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
rounds = np.arange(1, T + 1)

axes[0, 0].plot(cumulative_utility)
axes[0, 0].set_title("Cumulative utility (last trial)")
axes[0, 0].set_xlabel("Round")
axes[0, 0].set_ylabel("Utility")

axes[0, 1].plot(cumulative_cost, label="Cumulative cost")
axes[0, 1].axhline(budget, color="red", linestyle="--", label="Budget")
axes[0, 1].set_title("Cumulative cost (last trial)")
axes[0, 1].set_xlabel("Round")
axes[0, 1].set_ylabel("Cost")
axes[0, 1].legend()

axes[1, 0].plot(lambdas)
axes[1, 0].set_title("Dual variable (last trial)")
axes[1, 0].set_xlabel("Round")
axes[1, 0].set_ylabel("Lambda")

axes[1, 1].plot(average_utility, label="Average utility")
axes[1, 1].plot(average_cost, label="Average cost")
axes[1, 1].axhline(learner.rho, color="red", linestyle="--", label="Target cost per round")
axes[1, 1].set_title("Per-round averages (last trial)")
axes[1, 1].set_xlabel("Round")
axes[1, 1].legend()

axes[2, 0].plot(rounds, average_regret, label="Estimated regret")
axes[2, 0].fill_between(
    rounds,
    average_regret - regret_ci95,
    average_regret + regret_ci95,
    alpha=0.25,
    label="Uncertainty: 95% CI"
)
axes[2, 0].plot(rounds, regret, alpha=0.7, label="Last trial regret")
axes[2, 0].axhline(0.0, color="black", linewidth=1)
axes[2, 0].set_title("Regret vs piecewise-static oracle with uncertainty")
axes[2, 0].set_xlabel("Round")
axes[2, 0].set_ylabel("Regret")
axes[2, 0].legend()

axes[2, 1].plot(remaining_budgets, label="Learner remaining budget")
axes[2, 1].plot(oracle_remaining_budget, label="Oracle remaining budget")
axes[2, 1].axhline(0.0, color="red", linestyle="--")
axes[2, 1].set_title("Remaining budget (last trial)")
axes[2, 1].legend()
axes[2, 1].set_xlabel("Round")
axes[2, 1].set_ylabel("Budget")

plt.tight_layout()
plt.show()